# Import libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import re
from sklearn.metrics import cohen_kappa_score

# Path

In [ ]:
list_judges = ['openai/gpt-oss-20b', 'llama-3.3-70b-versatile', 'moonshotai/kimi-k2-instruct', 'qwen/qwen3-32b', 'tinyllama']

In [ ]:
def extract_model_name(model_string):
  match = re.match(r'[^/-]+', model_string)
  if match:
    return match.group(0)
  else:
    return model_string

In [ ]:
# Create list of dicts with judge name and corresponding input file path
judges_name_and_path = []
for judge in list_judges:
  judge_name = extract_model_name(judge)
  new_judge = {
      "name": judge_name,
      "path": f'drive/MyDrive/ViJudge/data/language_bias/{judge_name}_language_bias_for_pair_llama_deepseek.csv'
  }
  judges_name_and_path.append(new_judge)

In [ ]:
judges_name_and_path

In [ ]:
output_path = 'drive/MyDrive/ViJudge/data/language_bias/statistics/'

# Statistics

In [ ]:
def cal_measures_attempt(df: pd.DataFrame):
  # Compute Cohen's Kappa to measure agreement reliability
  # Pivot data so each row is a question_id and columns are verdicts per config
  pivot_df = df.pivot(index='question_id', columns='language_config', values='verdict')
  kappa_result = {}
  baseline = 'VN-VN' # Reference baseline
  configs_to_compare = ['EN-EN', 'VN2-VN2']
  for config in configs_to_compare:
    pair_data = pivot_df[[baseline, config]]
    kappa = cohen_kappa_score(pair_data[baseline], pair_data[config])
    kappa_result[f"{baseline} vs {config}"] = round(kappa, 4)

  return kappa_result

In [ ]:
kappa_results = []

for judge in judges_name_and_path:
  df = pd.read_csv(judge['path'])

  print(f"Judge: {judge['name']}")
  print(f"Total id: {len(df['question_id'].unique())}")
  print('*'*50)

  for attempt in df['attempt'].unique():
    attempt_df = df[df['attempt'] == attempt]
    kappa_dict = cal_measures_attempt(attempt_df)

    # Append to kappa_results (dict)
    new_kappa = {'judge': judge['name'], 'attempt': attempt}
    new_kappa.update(kappa_dict)
    kappa_results.append(new_kappa)

# Save & show result

In [ ]:
df_kappa_final = pd.DataFrame(kappa_results)
df_kappa_final.to_csv(output_path + 'kappa_result.csv', index=False)
df_kappa_final